In [1]:
import xarray as xr
import glob
import os
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 

### Download and Concatenate All 4 Files

In [2]:
data_dir = '../disdata/cems_fwi_data'
files = [
    os.path.join(data_dir, 'fwi_2002-05.nc'),
    os.path.join(data_dir, 'fwi_2006-11.nc'),
    os.path.join(data_dir, 'fwi_2012-17.nc'),
    os.path.join(data_dir, 'fwi_2018-23.nc'),
]
ds = xr.open_mfdataset(files, combine='nested', concat_dim='valid_time')

# Crop to core study region (36°S-39°S, 71°W-74°W)
# Check latitude ordering first: some ERA5 products store latitude descending (90 to -90)
print("Latitude order (first 5):", ds.latitude.values[:5])

ds = ds.sel(
    latitude=slice(-36, -39),
    longitude=slice(286, 289)
)

# Convert valid_time to datetime index
times = pd.to_datetime(ds.valid_time.values)
lats = ds.latitude.values
lons = ds.longitude.values

# Extract FWI as numpy array: shape (time, lat, lon)
fwi_array = ds['fwinx'].values
print(f"FWI array shape: {fwi_array.shape}")
print(f"Time steps: {len(times)}, Lats: {len(lats)}, Lons: {len(lons)}")
print(f"Date range: {times[0].date()} to {times[-1].date()}")

Latitude order (first 5): [-36.  -36.5 -37.  -37.5 -38. ]
FWI array shape: (8035, 7, 7)
Time steps: 8035, Lats: 7, Lons: 7
Date range: 2002-01-01 to 2023-12-31


In [3]:
for f in files:
    print(f"\n{'='*60}")
    print(f"File: {f}")
    print('='*60)
    ds_single = xr.open_dataset(f)
    print(ds_single)
    print(f"\nLat range: {ds_single.latitude.values.min()} to {ds_single.latitude.values.max()}")
    print(f"Lon range: {ds_single.longitude.values.min()} to {ds_single.longitude.values.max()}")
    print(f"Grid size: {len(ds_single.latitude)} x {len(ds_single.longitude)} = {len(ds_single.latitude)*len(ds_single.longitude)} cells")
    times_single = pd.to_datetime(ds_single.valid_time.values)
    print(f"Time range: {times_single[0]} to {times_single[-1]}")
    print(f"Number of time steps: {len(times_single)}")
    ds_single.close()


File: ../disdata/cems_fwi_data/fwi_2002-05.nc
<xarray.Dataset> Size: 544kB
Dimensions:     (valid_time: 1461, latitude: 7, longitude: 13)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 12kB 2002-01-01 ... 2005-12-31
    surface     float64 8B ...
  * latitude    (latitude) float64 56B -36.0 -36.5 -37.0 -37.5 -38.0 -38.5 -39.0
  * longitude   (longitude) float64 104B 285.0 285.5 286.0 ... 290.0 290.5 291.0
Data variables:
    fwinx       (valid_time, latitude, longitude) float32 532kB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-06-13T13:07 GRIB to CDM+CF via cfgrib-0.9.1...

Lat range: -39.0 to -36.0
Lon range: 285.0 to 291.0
Grid size: 7 x 13 = 91 cells
Time range: 2002-01-01 00:00:

In [4]:
for f in files:
    ds_single = xr.open_dataset(f)
    
    lats_single = ds_single.latitude.values
    lons_single = ds_single.longitude.values
    
    print(f"\nFile: {os.path.basename(f)}")
    print(f"  Latitudes ({len(lats_single)}): {sorted(lats_single)}")
    print(f"  Longitudes ({len(lons_single)}): {sorted(lons_single)}")
    print(f"  Lat range: {lats_single.min()} to {lats_single.max()}")
    print(f"  Lon range: {lons_single.min()} to {lons_single.max()}")
    
    # Check against core study region (36°S-39°S, 71°W-74°W)
    core_check = (lats_single.min() >= -39) and (lats_single.max() <= -36) and \
                 (lons_single.min() >= -74) and (lons_single.max() <= -71)
    print(f"  Within core study region (36°S-39°S, 71°W-74°W)?: {core_check}")
    
    ds_single.close()


File: fwi_2002-05.nc
  Latitudes (7): [np.float64(-39.0), np.float64(-38.5), np.float64(-38.0), np.float64(-37.5), np.float64(-37.0), np.float64(-36.5), np.float64(-36.0)]
  Longitudes (13): [np.float64(285.0), np.float64(285.5), np.float64(286.0), np.float64(286.5), np.float64(287.0), np.float64(287.5), np.float64(288.0), np.float64(288.5), np.float64(289.0), np.float64(289.5), np.float64(290.0), np.float64(290.5), np.float64(291.0)]
  Lat range: -39.0 to -36.0
  Lon range: 285.0 to 291.0
  Within core study region (36°S-39°S, 71°W-74°W)?: False

File: fwi_2006-11.nc
  Latitudes (7): [np.float64(-39.0), np.float64(-38.5), np.float64(-38.0), np.float64(-37.5), np.float64(-37.0), np.float64(-36.5), np.float64(-36.0)]
  Longitudes (13): [np.float64(285.0), np.float64(285.5), np.float64(286.0), np.float64(286.5), np.float64(287.0), np.float64(287.5), np.float64(288.0), np.float64(288.5), np.float64(289.0), np.float64(289.5), np.float64(290.0), np.float64(290.5), np.float64(291.0)]
  Lat 

### Calculate Per-Grid-Cell Percentile Thresholds

In [5]:
# Build a boolean mask for study period and non-zero FWI days
# for threshold calculation (July 2002 to June 2023)
study_mask = (times >= pd.Timestamp('2002-07-01')) & \
             (times <= pd.Timestamp('2023-06-30'))

# Initialise threshold arrays: shape (lat, lon)
p90_grid = np.full((len(lats), len(lons)), np.nan)
p95_grid = np.full((len(lats), len(lons)), np.nan)

print("Calculating per-cell thresholds...")

for i, lat in enumerate(lats):
    for j, lon in enumerate(lons):
        
        # Extract full time series for this grid cell
        cell_series = fwi_array[:, i, j]
        
        # Apply study period mask and exclude near-zero values
        cell_study = cell_series[study_mask]
        cell_nonzero = cell_study[cell_study > 1]
        
        if len(cell_nonzero) < 30:
            print(f"  Warning: few valid days at lat={lat}, lon={lon}")
            continue
        
        p90_grid[i, j] = np.percentile(cell_nonzero, 90)
        p95_grid[i, j] = np.percentile(cell_nonzero, 95)

print("\nPer-cell p90 thresholds:")
print(pd.DataFrame(p90_grid, index=lats, columns=lons).round(1))
print("\nPer-cell p95 thresholds:")
print(pd.DataFrame(p95_grid, index=lats, columns=lons).round(1))

Calculating per-cell thresholds...

Per-cell p90 thresholds:
       286.0  286.5  287.0  287.5  288.0  288.5  289.0
-36.0    NaN    NaN   13.5   34.7   44.9   33.0   35.0
-36.5    NaN   15.3   26.7   41.2   41.4   30.8   40.8
-37.0   23.2   19.2   31.4   42.3   35.8   29.1   46.4
-37.5    NaN   22.7   33.7   41.6   30.9   27.9   42.8
-38.0    NaN   19.2   30.0   34.6   24.4   29.2   43.5
-38.5   12.0   10.0   27.9   30.3   19.2   28.4   42.1
-39.0    NaN    8.6   27.2   25.4   16.3   24.1   38.1

Per-cell p95 thresholds:
       286.0  286.5  287.0  287.5  288.0  288.5  289.0
-36.0    NaN    NaN   16.0   39.9   52.5   36.5   39.0
-36.5    NaN   18.4   29.9   47.3   48.4   34.4   46.2
-37.0   28.4   23.2   34.5   49.4   41.6   35.2   53.8
-37.5    NaN   26.9   38.2   49.0   35.8   33.3   49.6
-38.0    NaN   22.3   33.8   40.8   29.1   35.0   49.7
-38.5   15.0   11.7   31.5   35.8   23.4   34.1   48.6
-39.0    NaN   10.2   31.1   30.1   20.0   28.1   43.7


### Define the consecutive-day season detection function

In [6]:
def detect_fire_season(daily_values, daily_dates, threshold, n_consecutive=3):
    """
    Detect fire season start and end using consecutive-day threshold crossing.
    Accepts numpy arrays for speed in the grid cell loop.
    """
    n = len(daily_values)
    
    qualifying_starts = []
    qualifying_ends = []
    
    i = 0
    while i <= n - n_consecutive:
        window = daily_values[i:i + n_consecutive]
        if np.all(window > threshold):
            run_start = i
            j = i + n_consecutive
            while j < n and daily_values[j] > threshold:
                j += 1
            run_end = j - 1
            qualifying_starts.append(run_start)
            qualifying_ends.append(run_end)
            i = j
        else:
            i += 1
    
    if len(qualifying_starts) == 0:
        return pd.NaT, pd.NaT, np.nan
    
    season_start = daily_dates[qualifying_starts[0]]
    season_end = daily_dates[qualifying_ends[-1]]
    season_length = (season_end - season_start).days + 1
    
    return season_start, season_end, season_length

### Apply per-grid-cell detection across all fire years & aggregate by year

In [7]:
fire_years = range(2003, 2024)

# Season length arrays
fsl_p90 = np.full((len(fire_years), len(lats), len(lons)), np.nan)
fsl_p95 = np.full((len(fire_years), len(lats), len(lons)), np.nan)

# Start/end day-of-fire-year arrays (day 1 = July 1st of the fire year)
start_doy_p90 = np.full((len(fire_years), len(lats), len(lons)), np.nan)
end_doy_p90 = np.full((len(fire_years), len(lats), len(lons)), np.nan)
start_doy_p95 = np.full((len(fire_years), len(lats), len(lons)), np.nan)
end_doy_p95 = np.full((len(fire_years), len(lats), len(lons)), np.nan)

print("Running per-cell season detection...")

for fy_idx, fy in enumerate(fire_years):
    
    fy_mask = (times >= pd.Timestamp(f'{fy-1}-07-01')) & \
              (times <= pd.Timestamp(f'{fy}-06-30'))
    
    fy_dates = times[fy_mask]
    fy_fwi = fwi_array[fy_mask, :, :]
    fy_start_date = pd.Timestamp(f'{fy-1}-07-01')
    
    for i in range(len(lats)):
        for j in range(len(lons)):
            
            cell_values = fy_fwi[:, i, j]
            
            s90, e90, l90 = detect_fire_season(
                cell_values, fy_dates, p90_grid[i, j], n_consecutive=3
            )
            fsl_p90[fy_idx, i, j] = l90
            if pd.notna(s90):
                start_doy_p90[fy_idx, i, j] = (s90 - fy_start_date).days + 1
                end_doy_p90[fy_idx, i, j] = (e90 - fy_start_date).days + 1
            
            s95, e95, l95 = detect_fire_season(
                cell_values, fy_dates, p95_grid[i, j], n_consecutive=3
            )
            fsl_p95[fy_idx, i, j] = l95
            if pd.notna(s95):
                start_doy_p95[fy_idx, i, j] = (s95 - fy_start_date).days + 1
                end_doy_p95[fy_idx, i, j] = (e95 - fy_start_date).days + 1
    
    n_detected_p90 = np.sum(~np.isnan(fsl_p90[fy_idx]))
    n_total_cells = len(lats) * len(lons)
    print(f"  Fire year {fy}: {n_detected_p90}/{n_total_cells} grid cells with detectable p90 season")

print("\nDone.")

Running per-cell season detection...
  Fire year 2003: 38/49 grid cells with detectable p90 season
  Fire year 2004: 39/49 grid cells with detectable p90 season
  Fire year 2005: 42/49 grid cells with detectable p90 season
  Fire year 2006: 35/49 grid cells with detectable p90 season
  Fire year 2007: 18/49 grid cells with detectable p90 season
  Fire year 2008: 34/49 grid cells with detectable p90 season
  Fire year 2009: 42/49 grid cells with detectable p90 season
  Fire year 2010: 27/49 grid cells with detectable p90 season
  Fire year 2011: 1/49 grid cells with detectable p90 season
  Fire year 2012: 39/49 grid cells with detectable p90 season
  Fire year 2013: 4/49 grid cells with detectable p90 season
  Fire year 2014: 43/49 grid cells with detectable p90 season
  Fire year 2015: 43/49 grid cells with detectable p90 season
  Fire year 2016: 43/49 grid cells with detectable p90 season
  Fire year 2017: 40/49 grid cells with detectable p90 season
  Fire year 2018: 42/49 grid cells 

In [8]:
# How many cells never got a valid threshold at all?
print("Cells with NaN p90 threshold:", np.sum(np.isnan(p90_grid)), "of", p90_grid.size)
print("Cells with NaN p95 threshold:", np.sum(np.isnan(p95_grid)), "of", p95_grid.size)

# Where are they?
nan_lat_idx, nan_lon_idx = np.where(np.isnan(p90_grid))
for i, j in zip(nan_lat_idx, nan_lon_idx):
    print(f"  lat={lats[i]}, lon={lons[j]}")

Cells with NaN p90 threshold: 6 of 49
Cells with NaN p95 threshold: 6 of 49
  lat=-36.0, lon=286.0
  lat=-36.0, lon=286.5
  lat=-36.5, lon=286.0
  lat=-37.5, lon=286.0
  lat=-38.0, lon=286.0
  lat=-39.0, lon=286.0


In [9]:
n_total_cells = len(lats) * len(lons)  # 49

fsl_p90_median = np.nanmedian(fsl_p90, axis=(1, 2))
fsl_p95_median = np.nanmedian(fsl_p95, axis=(1, 2))
fsl_p90_coverage = np.sum(~np.isnan(fsl_p90), axis=(1, 2)) / n_total_cells
fsl_p95_coverage = np.sum(~np.isnan(fsl_p95), axis=(1, 2)) / n_total_cells

start_p90_median = np.nanmedian(start_doy_p90, axis=(1, 2))
end_p90_median = np.nanmedian(end_doy_p90, axis=(1, 2))
start_p95_median = np.nanmedian(start_doy_p95, axis=(1, 2))
end_p95_median = np.nanmedian(end_doy_p95, axis=(1, 2))

fire_year_labels = [f"{fy-1}-{fy}" for fy in fire_years]
df_results = pd.DataFrame({
    'fire_year': fire_year_labels,
    'fsl_p90_median': fsl_p90_median,
    'fsl_p95_median': fsl_p95_median,
    'p90_cell_coverage': fsl_p90_coverage,
    'p95_cell_coverage': fsl_p95_coverage,
    'start_doy_p90': start_p90_median,
    'end_doy_p90': end_p90_median,
    'start_doy_p95': start_p95_median,
    'end_doy_p95': end_p95_median,
}).set_index('fire_year')

print(df_results.round(1))
print(f"\nYears where fewer than 50% of cells have detectable p90 season:")
print(df_results[df_results['p90_cell_coverage'] < 0.5].index.tolist())
print(f"\nYears where fewer than 50% of cells have detectable p95 season:")
print(df_results[df_results['p95_cell_coverage'] < 0.5].index.tolist())

           fsl_p90_median  fsl_p95_median  p90_cell_coverage  \
fire_year                                                      
2002-2003            40.0            31.5                0.8   
2003-2004            16.0             3.5                0.8   
2004-2005            12.5             3.0                0.9   
2005-2006             4.0             3.0                0.7   
2006-2007             3.0             4.0                0.4   
2007-2008            46.5             4.0                0.7   
2008-2009            68.5             5.0                0.9   
2009-2010            13.0             5.0                0.6   
2010-2011             3.0             NaN                0.0   
2011-2012            14.0             3.0                0.8   
2012-2013             3.5             3.5                0.1   
2013-2014            31.0            12.0                0.9   
2014-2015            78.0            54.0                0.9   
2015-2016            52.0             6.

/tmp/ipykernel_555557/767534364.py:4: RuntimeWarning: All-NaN slice encountered
  fsl_p95_median = np.nanmedian(fsl_p95, axis=(1, 2))
/tmp/ipykernel_555557/767534364.py:10: RuntimeWarning: All-NaN slice encountered
  start_p95_median = np.nanmedian(start_doy_p95, axis=(1, 2))
/tmp/ipykernel_555557/767534364.py:11: RuntimeWarning: All-NaN slice encountered
  end_p95_median = np.nanmedian(end_doy_p95, axis=(1, 2))


In [10]:
all_nan_p95_years = [
    fire_year_labels[fy_idx] for fy_idx in range(len(fire_years))
    if np.all(np.isnan(fsl_p95[fy_idx]))
]
print("Fire years with zero detectable p95 cells:", all_nan_p95_years)

Fire years with zero detectable p95 cells: ['2010-2011']


### Run Mann-Kendall on the median FSL series

In [11]:
import pymannkendall as mk

for label, col in [('p90 median FSL', 'fsl_p90_median'), 
                    ('p95 median FSL', 'fsl_p95_median')]:
    
    clean = df_results[col].dropna()
    
    print(f"\n{'='*50}")
    print(f"TREND ANALYSIS -- {label}")
    print(f"{'='*50}")
    print(f"Years included: {len(clean)}")
    
    if len(clean) < 4:
        print("Insufficient data for trend test")
        continue
    
    result = mk.original_test(clean)
    print(f"Mann-Kendall tau:   {result.Tau:.3f}")
    print(f"Mann-Kendall p:     {result.p:.4f}")
    print(f"Sen's slope:        {result.slope:.2f} days/year")
    print(f"Trend:              {result.trend}")


TREND ANALYSIS -- p90 median FSL
Years included: 21
Mann-Kendall tau:   0.371
Mann-Kendall p:     0.0200
Sen's slope:        2.17 days/year
Trend:              increasing

TREND ANALYSIS -- p95 median FSL
Years included: 20
Mann-Kendall tau:   0.405
Mann-Kendall p:     0.0130
Sen's slope:        0.38 days/year
Trend:              increasing


### Aggregate & Export 

In [12]:
df_results.to_csv('../outputs/method3_percell_results.csv')
print("Saved outputs/method3_percell_results.csv")
print(df_results.round(1))

Saved outputs/method3_percell_results.csv
           fsl_p90_median  fsl_p95_median  p90_cell_coverage  \
fire_year                                                      
2002-2003            40.0            31.5                0.8   
2003-2004            16.0             3.5                0.8   
2004-2005            12.5             3.0                0.9   
2005-2006             4.0             3.0                0.7   
2006-2007             3.0             4.0                0.4   
2007-2008            46.5             4.0                0.7   
2008-2009            68.5             5.0                0.9   
2009-2010            13.0             5.0                0.6   
2010-2011             3.0             NaN                0.0   
2011-2012            14.0             3.0                0.8   
2012-2013             3.5             3.5                0.1   
2013-2014            31.0            12.0                0.9   
2014-2015            78.0            54.0                0.9  